# 📓 Notebook 06 — Operator Console

Local abliterated model + MCP tool bridge + CLI agent + tunnel. The brain that drives the other notebooks.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 08:09:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB
overlay         236G   86G  151G  37% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED          
gpt-oss:20b         f38aa0c53da5    13 GB     49 minutes ago       
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    About an hour ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"11.6%","hits":397,"max_size":1000,"misses":3032,"size":21},"category_stats":{"additional":{"available":1,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":5,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":3,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":3032,"success_rate":"0.9%","system_metrics":{"cpu_percent":14.2,"disk_usage":36.1,"memory_percent":30.9,"network_io":{"bytes_recv":43254391521,"bytes_sent":176365973,"dropin":0,"dropout":0,"errin":0

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
curl -fsSL --connect-timeout 5 --max-time 15 https://opencode.ai/install 2>/dev/null | bash 2>/dev/null || npm i -g --timeout=15000 opencode-ai 2>/dev/null || true
opencode --version 2>/dev/null || echo "opencode agent ready"



Installing opencode version: 1.19.0
Successfully added opencode to $PATH in /root/.bashrc

                                 ▄     
█▀▀█ █▀▀█ █▀▀█ █▀▀▄ █▀▀▀ █▀▀█ █▀▀█ █▀▀█
█░░█ █░░█ █▀▀▀ █░░█ █░░░ █░░█ █░░█ █▀▀▀
▀▀▀▀ █▀▀▀ ▀▀▀▀ ▀  ▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀ ▀▀▀▀


OpenCode includes free models, to start:

cd <project>  # Open directory
opencode      # Run command

For more information visit https://opencode.ai/docs


opencode agent ready


In [ ]:
%%bash
# --- MCP servers beyond HexStrike ---
pip -q install mcp
git clone --depth 1 https://github.com/Eutectico/mcp-kali-server /content/mcp-kali 2>/dev/null || true
git clone --depth 1 https://github.com/sudohakan/kali-mcp-server /content/kali-mcp 2>/dev/null || true
echo "mcp servers cloned" 

mcp servers cloned


In [ ]:
%%bash
# --- hexstrike MCP frontend (stdio bridge) ---
ls /content/hexstrike/hexstrike_mcp.py && echo "mcp bridge present" 

/content/hexstrike/hexstrike_mcp.py
mcp bridge present


In [ ]:
import json, os
config = {
  "provider": {
    "local": {
      "npm": "@ai-sdk/openai-compatible",
      "options": { "baseURL": "http://localhost:11434/v1", "apiKey": "ollama" },
      "models": { "qwen3-coder-abl": { "name": "Qwen3-Coder-30B abliterated" } }
    }
  },
  "mcp": {
    "hexstrike": {
      "type": "local",
      "command": ["python3", "/content/hexstrike/hexstrike_mcp.py", "--server", "http://localhost:8888"]
    },
    "kali": {
      "type": "local",
      "command": ["python3", "/content/mcp-kali/server.py"]
    },
    "colab": {
      "type": "local",
      "command": ["uvx", "git+https://github.com/googlecolab/colab-mcp"]
    }
  }
}
os.makedirs("/root/.config/opencode", exist_ok=True)
with open("/root/.config/opencode/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("opencode mcp config saved successfully")


opencode mcp config saved successfully


In [ ]:
# Client connection instructions:
# uv tool install google-colab-cli
# colab ssh            # proxies 8888 (hexstrike) + 11434 (model) to localhost
print("Operator console ready for client connections.")


Operator console ready for client connections.